# NyayaAI Data Exploration

This notebook provides analysis and visualization of extracted case data.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
from src.config.settings import settings
from src.data_pipeline.loaders.db_loader import DatabaseLoader

# Set plot style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Connect to Database

In [ ]:
# Create database connection
engine = create_engine(settings.database_url)
db_loader = DatabaseLoader()

# Get statistics
stats = db_loader.get_database_stats()
print("Database Statistics:")
for key, value in stats.items():
    print(f"  {key}: {value}")

## 2. Load Case Data

In [ ]:
# Load all cases into DataFrame
query = "SELECT * FROM cases"
df = pd.read_sql(query, engine)

print(f"Loaded {len(df)} cases")
print(f"\nColumns: {df.columns.tolist()}")
df.head()

## 3. Data Quality Overview

In [ ]:
# Validation status distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Status pie chart
status_counts = df['validation_status'].value_counts()
axes[0].pie(status_counts.values, labels=status_counts.index, autopct='%1.1f%%')
axes[0].set_title('Validation Status Distribution')

# Confidence distribution
axes[1].hist(df['average_confidence'].dropna(), bins=20, edgecolor='black')
axes[1].set_xlabel('Average Confidence Score')
axes[1].set_ylabel('Number of Cases')
axes[1].set_title('Confidence Score Distribution')
axes[1].axvline(df['average_confidence'].mean(), color='red', linestyle='--', label=f'Mean: {df["average_confidence"].mean():.2f}')
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Court Distribution

In [ ]:
# Cases by court
court_counts = df['court'].value_counts().head(10)

plt.figure(figsize=(12, 6))
court_counts.plot(kind='barh')
plt.xlabel('Number of Cases')
plt.ylabel('Court')
plt.title('Top 10 Courts by Case Count')
plt.tight_layout()
plt.show()

print(f"\nTotal courts: {df['court'].nunique()}")

## 5. Case Types

In [ ]:
# Case type distribution
case_type_counts = df['case_type'].value_counts()

plt.figure(figsize=(10, 6))
case_type_counts.plot(kind='bar', color='steelblue')
plt.xlabel('Case Type')
plt.ylabel('Number of Cases')
plt.title('Distribution by Case Type')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 6. Verdict Analysis

In [ ]:
# Verdict distribution
verdict_counts = df['verdict'].value_counts()

plt.figure(figsize=(12, 6))
verdict_counts.plot(kind='bar', color='coral')
plt.xlabel('Verdict')
plt.ylabel('Number of Cases')
plt.title('Verdict Distribution')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 7. Temporal Analysis

In [ ]:
# Convert date column
df['date_of_judgment'] = pd.to_datetime(df['date_of_judgment'])
df['year'] = df['date_of_judgment'].dt.year
df['month'] = df['date_of_judgment'].dt.month

# Cases by year
year_counts = df['year'].value_counts().sort_index()

plt.figure(figsize=(12, 6))
year_counts.plot(kind='line', marker='o', linewidth=2)
plt.xlabel('Year')
plt.ylabel('Number of Cases')
plt.title('Cases Over Time')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Field Completeness

In [ ]:
# Calculate completeness for key fields
fields = ['case_title', 'court', 'date_of_judgment', 'case_type', 'verdict', 
          'petitioner', 'respondent', 'key_arguments_petitioner', 'court_reasoning']

completeness = {}
for field in fields:
    if field in df.columns:
        completeness[field] = (df[field].notna().sum() / len(df)) * 100

completeness_df = pd.DataFrame(list(completeness.items()), columns=['Field', 'Completeness %'])
completeness_df = completeness_df.sort_values('Completeness %', ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(completeness_df['Field'], completeness_df['Completeness %'], color='lightgreen')
plt.xlabel('Completeness (%)')
plt.ylabel('Field')
plt.title('Field Completeness')
plt.xlim(0, 100)
for i, v in enumerate(completeness_df['Completeness %']):
    plt.text(v + 1, i, f'{v:.1f}%', va='center')
plt.tight_layout()
plt.show()

## 9. Most Cited Statutes

In [ ]:
# Query statutes with case counts
statute_query = """
SELECT s.statute_name, COUNT(cs.case_id) as case_count
FROM statutes s
JOIN case_statutes cs ON s.id = cs.statute_id
GROUP BY s.statute_name
ORDER BY case_count DESC
LIMIT 15
"""

statutes_df = pd.read_sql(statute_query, engine)

if not statutes_df.empty:
    plt.figure(figsize=(12, 6))
    plt.barh(statutes_df['statute_name'], statutes_df['case_count'], color='skyblue')
    plt.xlabel('Number of Cases')
    plt.ylabel('Statute')
    plt.title('Most Frequently Cited Statutes')
    plt.tight_layout()
    plt.show()
else:
    print("No statute data available")

## 10. Low Confidence Cases

In [ ]:
# Cases with low confidence
low_confidence = df[df['average_confidence'] < 0.6][['case_id', 'case_title', 'court', 'average_confidence', 'validation_status']]

print(f"Cases with confidence < 0.6: {len(low_confidence)}\n")
if not low_confidence.empty:
    print(low_confidence.head(10))
else:
    print("No low confidence cases!")

## 11. Sample Case Review

In [ ]:
# Display a sample case in detail
if len(df) > 0:
    sample_case = df.iloc[0]
    
    print("SAMPLE CASE DETAILS")
    print("=" * 80)
    print(f"Case ID: {sample_case['case_id']}")
    print(f"Title: {sample_case['case_title']}")
    print(f"Court: {sample_case['court']}")
    print(f"Date: {sample_case['date_of_judgment']}")
    print(f"Type: {sample_case['case_type']}")
    print(f"Verdict: {sample_case['verdict']}")
    print(f"Confidence: {sample_case['average_confidence']:.2f}")
    print(f"Validation: {sample_case['validation_status']}")
    print(f"\nBench: {sample_case['bench']}")
    print(f"\nPetitioner: {sample_case['petitioner']}")
    print(f"Respondent: {sample_case['respondent']}")
    
    if sample_case['court_reasoning']:
        print(f"\nCourt Reasoning (excerpt):")
        print(sample_case['court_reasoning'][:500] + "...")

## 12. Summary Statistics

In [ ]:
print("SUMMARY STATISTICS")
print("=" * 80)
print(f"Total Cases: {len(df)}")
print(f"Unique Courts: {df['court'].nunique()}")
print(f"Date Range: {df['date_of_judgment'].min()} to {df['date_of_judgment'].max()}")
print(f"Average Confidence: {df['average_confidence'].mean():.3f}")
print(f"Median Confidence: {df['average_confidence'].median():.3f}")
print(f"\nValidation Status:")
print(df['validation_status'].value_counts())
print(f"\nCase Types:")
print(df['case_type'].value_counts())
print(f"\nVerdicts:")
print(df['verdict'].value_counts())